## Sequence to Sequence Translation


In [62]:
#translation-encoder.txt
#translation-transformer.txt

In [63]:
import numpy as np
import tensorflow as tf
from keras.models import Model
from keras.layers import Input,LSTM,Dense

In [64]:
data_path =  [
    ("Go.", "Va !"),
    ("Run!", "Cours !"),
    ("Run.", "Cours !"),
    ("Who?", "Qui ?"),
    ("Wow!", "Ça alors !"),
    ("Fire!", "Au feu !"),
    ("Help!", "À l'aide !"),
    ("Stop!", "Arrête-toi !"),
    ("Wait!", "Attends !"),
    ("Hello!", "Bonjour !"),
    ("I see.", "Je comprends.")

]

## Data Preprocessing

In [65]:
input_texts  = [] #character level sequence to sequence 
target_texts = []

input_characters = set()
target_characters = set()

for input_text, target_text in data_path:
  target_text = '\t' + target_text + '\n'
  input_texts.append(input_text)
  target_texts.append(target_text)

  for char in input_text:
    if char not in input_characters:
      input_characters.add(char)

  for char in target_text:
    if char not in target_characters:
      target_characters.add(char)


In [66]:
input_texts

['Go.',
 'Run!',
 'Run.',
 'Who?',
 'Wow!',
 'Fire!',
 'Help!',
 'Stop!',
 'Wait!',
 'Hello!',
 'I see.']

In [67]:
target_texts

['\tVa !\n',
 '\tCours !\n',
 '\tCours !\n',
 '\tQui ?\n',
 '\tÇa alors !\n',
 '\tAu feu !\n',
 "\tÀ l'aide !\n",
 '\tArrête-toi !\n',
 '\tAttends !\n',
 '\tBonjour !\n',
 '\tJe comprends.\n']

In [68]:
input_characters = sorted(list(input_characters))


In [69]:
input_characters

[' ',
 '!',
 '.',
 '?',
 'F',
 'G',
 'H',
 'I',
 'R',
 'S',
 'W',
 'a',
 'e',
 'h',
 'i',
 'l',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'w']

In [70]:
target_characters = sorted(list(target_characters))

In [71]:
target_characters

['\t',
 '\n',
 ' ',
 '!',
 "'",
 '-',
 '.',
 '?',
 'A',
 'B',
 'C',
 'J',
 'Q',
 'V',
 'a',
 'c',
 'd',
 'e',
 'f',
 'i',
 'j',
 'l',
 'm',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'À',
 'Ç',
 'ê']

In [72]:
num_decoder_tokens = len(target_characters)
num_encoder_tokens = len(input_characters)

In [73]:
num_decoder_tokens,num_encoder_tokens

(33, 24)

In [74]:
max_encoder_seq_length = max([len(x) for x in input_texts])
max_decoder_seq_length = max([len(x) for x in target_texts])

In [75]:
max_encoder_seq_length,max_decoder_seq_length

(6, 15)

## Create token mappings(char-> int)

In [76]:
input_token_index = dict([char,i] for i, char in enumerate(input_characters))
target_token_index = dict([char,i] for i, char in enumerate(target_characters))

In [77]:
target_token_index

{'\t': 0,
 '\n': 1,
 ' ': 2,
 '!': 3,
 "'": 4,
 '-': 5,
 '.': 6,
 '?': 7,
 'A': 8,
 'B': 9,
 'C': 10,
 'J': 11,
 'Q': 12,
 'V': 13,
 'a': 14,
 'c': 15,
 'd': 16,
 'e': 17,
 'f': 18,
 'i': 19,
 'j': 20,
 'l': 21,
 'm': 22,
 'n': 23,
 'o': 24,
 'p': 25,
 'r': 26,
 's': 27,
 't': 28,
 'u': 29,
 'À': 30,
 'Ç': 31,
 'ê': 32}

## Create reverse token mappings(int-> char)

In [78]:
reverse_input_token_index = dict([i,char] for i, char in enumerate(input_characters))
reverse_target_token_index = dict([i,char] for i, char in enumerate(target_characters))

In [79]:
reverse_input_token_index

{0: ' ',
 1: '!',
 2: '.',
 3: '?',
 4: 'F',
 5: 'G',
 6: 'H',
 7: 'I',
 8: 'R',
 9: 'S',
 10: 'W',
 11: 'a',
 12: 'e',
 13: 'h',
 14: 'i',
 15: 'l',
 16: 'n',
 17: 'o',
 18: 'p',
 19: 'r',
 20: 's',
 21: 't',
 22: 'u',
 23: 'w'}

In [80]:
encoder_data_input = np.zeros((
    len(input_texts),
    max_encoder_seq_length,
    num_encoder_tokens
))
encoder_data_input

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
    

In [81]:
encoder_data_input.shape

(11, 6, 24)

In [82]:
decoder_data_input = np.zeros((
    len(input_texts),
    max_decoder_seq_length,
    num_decoder_tokens
))
decoder_data_input

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.

In [83]:
decoder_data_input.shape

(11, 15, 33)

In [84]:
decoder_target_data = np.zeros((
    len(input_texts),
    max_decoder_seq_length,
    num_decoder_tokens
))


In [85]:
decoder_target_data.shape

(11, 15, 33)

In [86]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
  for t, char in enumerate(input_text):
    encoder_data_input[i, t, input_token_index[char]] = 1
  encoder_data_input[i, t + 1:, input_token_index[" "]] = 1  # padding

  for t, char in enumerate(target_text):  
    decoder_data_input[i, t, target_token_index[char]] = 1
    if t > 0:    
      decoder_target_data[i, t - 1, target_token_index[char]] = 1
  decoder_data_input[i, t + 1:, target_token_index[" "]] = 1  # padding
  decoder_target_data[i, t:, target_token_index[" "]] = 1  # padding


In [87]:
decoder_target_data

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0.

In [88]:
decoder_target_data.shape

(11, 15, 33)

## Build the model

In [89]:
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)

In [90]:
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=[state_h,state_c])

decoder_dense = Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

## define the model

first layer is encoder inputs
second layar is decoder input
third layer is decoder dense
final layer is decoder outputs

In [91]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [92]:
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])


In [93]:
history=model.fit([encoder_data_input, decoder_data_input], decoder_target_data,epochs=500)

Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.0000e+00 - loss: 3.5170
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.4364 - loss: 3.4277
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4485 - loss: 3.3410
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4485 - loss: 3.1968
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.4485 - loss: 2.8289
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.4485 - loss: 2.2456
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.4485 - loss: 2.1849
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.4485 - loss: 2.1389
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.4485 - loss: 2.1026
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4485 - loss: 2.0829
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.4485 - loss: 2.1043
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4485 - 

In [94]:
encoder_model=Model(encoder_inputs, [state_h,state_c])


In [95]:
decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, decoder_state_h, decoder_state_c = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs, decoder_state_h, decoder_state_c]
)

print("Inference models created successfully!")


Inference models created successfully!


In [96]:
def translate_sentence(input_sentence, max_output_length=None):
    if max_output_length is None:
        max_output_length = max_decoder_seq_length

    encoder_input = np.zeros(
        (1, max_encoder_seq_length, num_encoder_tokens), dtype="float32"
    )

    for t, char in enumerate(input_sentence):
        if t >= max_encoder_seq_length:
            break  # ignore extra characters beyond what the model was trained on
        if char in input_token_index:
            encoder_input[0, t, input_token_index[char]] = 1.0
        else:
            print(f"Warning: unknown character '{char}' skipped (treated as padding).")
            encoder_input[0, t, input_token_index[" "]] = 1.0

    states_value = encoder_model.predict(encoder_input, verbose=0)

    target_seq = np.zeros((1, 1, num_decoder_tokens), dtype="float32")
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    decoded_sentence = ""

    for _ in range(max_output_length):
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_token_index[sampled_token_index]

       
        if sampled_char == "\n":
            break

        
        if sampled_char != "\t":
            decoded_sentence += sampled_char

   
        target_seq = np.zeros((1, 1, num_decoder_tokens), dtype="float32")
        target_seq[0, 0, sampled_token_index] = 1.0

   
        states_value = [h, c]

    return decoded_sentence


In [97]:

result=translate_sentence("Go.")
print("English :", "Go.")
print("French :", result)
print()
print(result)

"""
print(f"{'English':<10} | {'Expected French':<15} | {'Model output'}")
print("-" * 45)
for eng, fr in data_path:
prediction = translate_sentence(eng)
print(f"{eng:<10} | {fr:<15} | {prediction}")
"""

English : Go.
French : Va !

Va !


'\nprint(f"{\'English\':<10} | {\'Expected French\':<15} | {\'Model output\'}")\nprint("-" * 45)\nfor eng, fr in data_path:\nprediction = translate_sentence(eng)\nprint(f"{eng:<10} | {fr:<15} | {prediction}")\n'